# Multi-objective optimization (v4)

本版相对 v3 的修复：
1. **特征名大小写对齐**：`'building density'` → `'Building density'`（与训练逐字一致），避免 assert 报错 / 静默零填充。
2. **`MODEL_FEATURES` 直接读集成对象的 `feature_names_in_`**（兼容 CatBoost/XGB/LGBM 任意为首，带回退）。
3. **后处理预测块移出 `for f in faces` 循环**：每个解只预测一次（原来重复 5 次），并改用 assert 取代静默零填充。
4. 清理死代码与过时注释。

> 说明：**SVF 现已随形态变化**——在加载数据时用训练集拟合 `SVF = f(几何特征)`（随机森林，每栋楼一行），优化时按候选形态实时预测并 clip 到训练范围。

In [2]:
import os
import numpy as np
import pandas as pd
from joblib import load
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.optimize import minimize
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import parallel_coordinates
import warnings

warnings.filterwarnings('ignore')

In [3]:
# ==================== 0. 核心类定义 ====================
# 必须与训练脚本中的 WeightedEnsemble 同名同结构，joblib 才能正确反序列化。
class WeightedEnsemble:
    def __init__(self, models, weights, feature_names=None):
        self.models = list(models)
        self.weights = list(weights)
        if feature_names is not None:
            self.feature_names_in_ = np.asarray(feature_names, dtype=object)
            self.feature_name_ = list(feature_names)
        else:
            self.feature_names_in_ = None
            self.feature_name_ = None

    def predict(self, X):
        preds = [np.asarray(m.predict(X)).ravel() for m in self.models]
        return np.average(preds, axis=0, weights=self.weights)

In [4]:
# ==================== 1. 配置参数 ====================
MODEL_DIR = r'D:\research_paper\photovoltaic_prediction\training_results\model'
RESULT_DIR = r'D:\research_paper\photovoltaic_prediction\optimization_result'
DATA_PATH = r'D:\research_paper\photovoltaic_prediction\all_cleaned_data.csv'
os.makedirs(RESULT_DIR, exist_ok=True)

ELECTRICITY_PRICE = 0.950  # 购电价 (元/kWh)
FEED_IN_TARIFF = 0.753     # 上网价 (元/kWh)
PV_COST_PER_M2_ROOF = 950  # 屋顶铺设单价 (元/m²)
PV_COST_PER_M2_WALL = 950  # 立面铺设单价 (元/m²)
MAINTENANCE_RATE = 0.015   # 年维护费率
DISCOUNT_RATE = 0.04       # 折现率
LIFESPAN = 20              # 项目寿命 (年)
EFFICIENCY_LOSS = 0.01     # 每年衰减率 (1%)

# 碳排放因子 (g CO2-eq/kWh)
GRID_CARBON_FACTOR = 683.5
PV_CARBON_FACTOR = 14.2

In [5]:
# ==================== 2. 模型加载 ====================
print("正在加载集成模型...")
TARGETS = ['hourly_EUI', 'hourly_power_generation_roof', 'hourly_power_generation_east',
           'hourly_power_generation_south', 'hourly_power_generation_west', 'hourly_power_generation_north']
models = {t: load(os.path.join(MODEL_DIR, f"best_model_Ensemble_Best_{t}.joblib")) for t in TARGETS}

# 直接读集成对象暴露的特征名与顺序（最稳）；旧模型回退到首个基模型
try:
    MODEL_FEATURES = list(models['hourly_EUI'].feature_names_in_)
except (AttributeError, TypeError):
    m0 = models['hourly_EUI'].models[0]
    MODEL_FEATURES = (list(m0.feature_name_) if hasattr(m0, 'feature_name_')
                      else list(m0.feature_names_in_))

print(f"成功对接模型！期望特征数量: {len(MODEL_FEATURES)}")
print(f"特征顺序: {MODEL_FEATURES}")

正在加载集成模型...
成功对接模型！期望特征数量: 24
特征顺序: ['FAR', 'Building density', 'Building Height', 'Building Amount', 'Shape Factor', 'depth', 'frontage', 'rotation', 'SVF', 'roof_to_envelope_ratio', 'roof_to_floor_ratio', 'OSR', 'dry_bulb_temperature', 'relative_humidity', 'DNI', 'DHI', 'GHI', 'hour', 'month', 'day_of_year', 'is_daylight', 'cos_solar_alt', 'main_type_encoded', 'code_encoded']


In [7]:
# ==================== 3. 全局数据处理与 SVF 拟合 (仅执行一次) ====================
# [注意]：这里的代码放在循环外面，避免重复读取和重复训练随机森林
real_config = {
    'PLOT_AREA': 62500,
    'FLOOR_HEIGHT': 3.0,
    'PV_COEFF': {'roof': 0.6, 'wall': 0.53}
}

CODE_CONSTRAINTS = {}
for m_idx in [0, 1, 2]:
    for c_idx in [0, 1, 2]:
        if c_idx == 0: far_range = (1.0, 2.5)
        elif c_idx == 1: far_range = (1.5, 4.0)
        else: far_range = (2.5, 6.0)

        if m_idx == 0: d_range, f_range = (12.0, 24.0), (30.0, 60.0)
        elif m_idx == 1: d_range, f_range = (10.0, 20.0), (20.0, 40.0)
        else: d_range, f_range = (15.0, 30.0), (30.0, 45.0)
        CODE_CONSTRAINTS[(m_idx, c_idx)] = {'FAR': far_range, 'depth': d_range, 'frontage': f_range}

AMOUNT_CONSTRAINTS = {
    (0, 0): (6, 24), (0, 1): (4, 20), (0, 2): (4, 20),
    (1, 0): (9, 25), (1, 1): (4, 16), (1, 2): (4, 12),
    (2, 0): (4, 16), (2, 1): (4, 16), (2, 2): (4, 16)
}

DENSITY_LIMITS = {0: (0.01, 0.35), 1: (0.01, 0.30), 2: (0.01, 0.25)}

if os.path.exists(DATA_PATH):
    df_raw = pd.read_csv(DATA_PATH)
    first_file = df_raw['source_file'].unique()[0]

    footprint = df_raw['frontage'] * df_raw['depth'] * df_raw['Building Amount']
    df_raw['Building density'] = footprint / real_config['PLOT_AREA']
    df_raw['main_type'] = df_raw['building_type'].apply(lambda x: str(x).split('_')[0])
    df_raw['code_classification'] = df_raw['building_type'].apply(lambda x: "_".join(str(x).split('_')[1:]) if '_' in str(x) else 'Unknown')

    le_main, le_code = LabelEncoder(), LabelEncoder()
    df_raw['main_type_encoded'] = le_main.fit_transform(df_raw['main_type'])
    df_raw['code_encoded'] = le_code.fit_transform(df_raw['code_classification'])

    ws = df_raw[df_raw['source_file'] == first_file].sort_values('date').copy()
    ws['timestamp'] = pd.to_datetime(ws['date'])

    bg_weather_matrix = np.column_stack([
        ws['dry_bulb_temperature'], ws['relative_humidity'], ws['DNI'], ws['DHI'], ws['GHI'],
        ws['timestamp'].dt.hour, ws['timestamp'].dt.month, ws['timestamp'].dt.dayofyear,
        (ws['GHI'] > 0).astype(int), np.cos(np.radians(ws['Solar Altitude'].fillna(0).clip(lower=0)))
    ])

    from sklearn.ensemble import RandomForestRegressor
    SVF_GEO_COLS = ['FAR', 'Building density', 'Building Height', 'Building Amount',
                    'Shape Factor', 'depth', 'frontage', 'main_type_encoded', 'code_encoded', 'rotation']
    _svf_tbl = df_raw.dropna(subset=['SVF']).drop_duplicates('source_file')
    svf_model = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)
    svf_model.fit(_svf_tbl[SVF_GEO_COLS].astype(float), _svf_tbl['SVF'].astype(float))
    SVF_MIN, SVF_MAX = float(_svf_tbl['SVF'].min()), float(_svf_tbl['SVF'].max())
    print(f"全局 SVF 回归器已拟合（{len(_svf_tbl)} 栋楼）；训练范围 [{SVF_MIN:.3f}, {SVF_MAX:.3f}]")
else:
    raise FileNotFoundError(f"未找到数据文件: {DATA_PATH}")

def predict_svf_batch(geom_rows):
    df = pd.DataFrame(np.asarray(geom_rows, dtype=float), columns=SVF_GEO_COLS)
    return np.clip(svf_model.predict(df), SVF_MIN, SVF_MAX)

def predict_svf(*features):
    return float(predict_svf_batch([list(features)])[0])

全局 SVF 回归器已拟合（843 栋楼）；训练范围 [27.139, 49.449]


## SVF_validation：SVF ~ 形态 拟合优度

用 5 折交叉验证评估"由形态特征重建 SVF"的可靠性（R² / MAE / RMSE + 实测-预测散点）。这一段可作为论文中 SVF 重建合理性的佐证。

In [9]:
# ==================== 4. 核心功能函数定义 ====================
# [注意]：由于外部会动态修改 FLOOR_BOUNDS，函数内部通过传参或动态捕获的方式读取
def get_full_feature_vector(p_ind, floor_bounds):
    h_floor = int(np.clip(np.round(p_ind[0]), floor_bounds[0], floor_bounds[1]))
    h_far_pct = float(p_ind[1])
    h_depth_pct = float(p_ind[2])
    h_front_pct = float(p_ind[3])
    h_rot = float(p_ind[4])
    h_main_idx = int(np.clip(np.round(p_ind[5]), 0, 2))
    h_code = 0 if h_floor <= 9 else (1 if h_floor <= 18 else 2)
    limits = CODE_CONSTRAINTS[(h_main_idx, h_code)]
    real_far_target = limits['FAR'][0] + h_far_pct * (limits['FAR'][1] - limits['FAR'][0])
    real_depth = limits['depth'][0] + h_depth_pct * (limits['depth'][1] - limits['depth'][0])
    real_front = limits['frontage'][0] + h_front_pct * (limits['frontage'][1] - limits['frontage'][0])
    h_height = h_floor * real_config['FLOOR_HEIGHT']
    plot_area = real_config['PLOT_AREA']
    h_amt_raw = (real_far_target * plot_area) / (real_depth * real_front * h_floor + 1e-6)
    amt_min, amt_max = AMOUNT_CONSTRAINTS[(h_main_idx, h_code)]
    h_amt_val = np.clip(np.round(h_amt_raw), amt_min, amt_max)
    dens_min, dens_max = DENSITY_LIMITS[h_code]
    h_density_current = (real_depth * real_front * h_amt_val) / plot_area
    if h_density_current > dens_max:
        h_amt_val = np.floor((dens_max * plot_area) / (real_depth * real_front + 1e-6))
        h_amt_val = np.maximum(h_amt_val, amt_min)
    real_far_actual = (h_amt_val * real_depth * real_front * h_floor) / plot_area
    s_roof = real_depth * real_front
    s_walls = 2 * (real_depth + real_front) * h_height
    shape_factor = (s_roof + s_walls) / (s_roof * h_height + 1e-6)
    b_feat = [real_far_actual, h_height, h_amt_val, shape_factor, real_depth, real_front, h_rot]
    c_feat = [h_main_idx, h_code]
    return b_feat, c_feat, h_amt_val

def compute_face_area(face_name, frontage, depth, height, building_amount):
    coeff_roof = real_config['PV_COEFF']['roof']
    coeff_wall = real_config['PV_COEFF']['wall']
    if face_name == 'roof': return frontage * depth * building_amount * coeff_roof
    elif face_name in ['east', 'west']: return depth * height * building_amount * coeff_wall
    elif face_name in ['south', 'north']: return frontage * height * building_amount * coeff_wall
    return 0.0

def build_feature_pool(num_hours, far_actual, building_height, amount, shape_factor,
                       depth, front, rotation, h_main_idx, h_code, svf_val):
    a_roof = front * depth * amount
    a_wall_ew = 2.0 * (depth * building_height * amount)
    a_wall_sn = 2.0 * (front * building_height * amount)
    a_env = a_roof + a_wall_ew + a_wall_sn
    building_density = a_roof / real_config['PLOT_AREA']
    return {
        'FAR': np.full(num_hours, far_actual), 'Building density': np.full(num_hours, building_density),
        'Building Height': np.full(num_hours, building_height), 'Building Amount': np.full(num_hours, amount),
        'Shape Factor': np.full(num_hours, shape_factor), 'depth': np.full(num_hours, depth),
        'frontage': np.full(num_hours, front), 'rotation': np.full(num_hours, rotation),
        'SVF': np.full(num_hours, svf_val), 'roof_to_envelope_ratio': np.full(num_hours, a_roof / (a_env + 1e-9)),
        'roof_to_floor_ratio': np.full(num_hours, 1.0 / (building_height / real_config['FLOOR_HEIGHT'] + 1e-9)),
        'OSR': np.full(num_hours, (real_config['PLOT_AREA'] - a_roof) / real_config['PLOT_AREA']),
        'dry_bulb_temperature': bg_weather_matrix[:, 0], 'relative_humidity': bg_weather_matrix[:, 1],
        'DNI': bg_weather_matrix[:, 2], 'DHI': bg_weather_matrix[:, 3], 'GHI': bg_weather_matrix[:, 4],
        'hour': bg_weather_matrix[:, 5], 'month': bg_weather_matrix[:, 6], 'day_of_year': bg_weather_matrix[:, 7],
        'is_daylight': bg_weather_matrix[:, 8], 'cos_solar_alt': bg_weather_matrix[:, 9],
        'main_type_encoded': np.full(num_hours, h_main_idx), 'code_encoded': np.full(num_hours, h_code),
    }

def to_model_matrix(feature_pool):
    df_features = pd.DataFrame(feature_pool)
    missing = [c for c in MODEL_FEATURES if c not in df_features.columns]
    assert not missing, f"❌ 特征名不匹配，会被零填充: {missing}"
    return df_features[MODEL_FEATURES].values.astype(np.float32)

def resolve_design(p_ind, floor_bounds):
    h_floor = int(np.clip(np.round(p_ind[0]), floor_bounds[0], floor_bounds[1]))
    h_main_idx = int(np.clip(np.round(p_ind[5]), 0, 2))
    h_code = 0 if h_floor <= 9 else (1 if h_floor <= 18 else 2)
    limits = CODE_CONSTRAINTS[(h_main_idx, h_code)]
    far_target = limits['FAR'][0] + float(p_ind[1]) * (limits['FAR'][1] - limits['FAR'][0])
    depth = limits['depth'][0] + float(p_ind[2]) * (limits['depth'][1] - limits['depth'][0])
    front = limits['frontage'][0] + float(p_ind[3]) * (limits['frontage'][1] - limits['frontage'][0])
    plot_area = real_config['PLOT_AREA']
    building_height = h_floor * real_config['FLOOR_HEIGHT']
    single_area = depth * front
    amount_ideal = (far_target * plot_area) / (single_area * h_floor + 1e-6)
    amount_min, amount_max = AMOUNT_CONSTRAINTS[(h_main_idx, h_code)]
    amount = np.clip(np.round(amount_ideal), amount_min, amount_max)
    density = (single_area * amount) / plot_area
    dens_min, dens_max = DENSITY_LIMITS[h_code]
    if density < dens_min: amount = np.clip(np.ceil(dens_min * plot_area / single_area), amount_min, amount_max)
    elif density > dens_max: amount = np.clip(np.floor(dens_max * plot_area / single_area), amount_min, amount_max)
    far_actual = (amount * single_area * h_floor) / plot_area
    shape_factor = (single_area + 2 * (depth + front) * building_height) / (single_area * building_height + 1e-6)
    building_density = (front * depth * amount) / plot_area
    return {'h_floor': h_floor, 'h_main_idx': h_main_idx, 'h_code': h_code,
            'far_actual': far_actual, 'depth': depth, 'front': front,
            'building_height': building_height, 'single_area': single_area,
            'amount': amount, 'rotation': float(p_ind[4]),
            'shape_factor': shape_factor, 'building_density': building_density}

def svf_geom_row(d):
    return [d['far_actual'], d['building_density'], d['building_height'], d['amount'],
            d['shape_factor'], d['depth'], d['front'], d['h_main_idx'], d['h_code'], d['rotation']]

def objective_func(p_ind, floor_bounds, svf_val=None):
    d = resolve_design(p_ind, floor_bounds)
    if svf_val is None: svf_val = predict_svf(*svf_geom_row(d))
    num_hours = bg_weather_matrix.shape[0]
    fp = build_feature_pool(num_hours, d['far_actual'], d['building_height'], d['amount'],
                            d['shape_factor'], d['depth'], d['front'], d['rotation'],
                            d['h_main_idx'], d['h_code'], svf_val)
    X = to_model_matrix(fp)
    eui_pred = np.maximum(models['hourly_EUI'].predict(X), 0)
    gen = {t.split('_')[-1]: np.maximum(models[t].predict(X), 0) for t in TARGETS[1:]}
    load_h = eui_pred * (d['single_area'] * d['h_floor'] * d['amount'])
    total_gen_h = sum(gen.values())
    total_ann_gen = total_gen_h.sum() + 1e-9
    total_ann_self = np.sum(np.minimum(load_h, total_gen_h))
    sc = total_ann_self / total_ann_gen
    er = total_ann_gen * (GRID_CARBON_FACTOR - PV_CARBON_FACTOR) / 1e6
    total_ann_export = np.maximum(0, total_ann_gen - total_ann_self)
    revenue_annual = total_ann_self * ELECTRICITY_PRICE + total_ann_export * FEED_IN_TARIFF
    total_capex = sum([compute_face_area(f, d['front'], d['depth'], d['building_height'], d['amount']) * (
        PV_COST_PER_M2_ROOF if f == 'roof' else PV_COST_PER_M2_WALL)
        for f in ['roof', 'east', 'south', 'west', 'north']])
    total_opex_annual = total_capex * MAINTENANCE_RATE
    npv = -total_capex
    decay_rate = 1.0 - EFFICIENCY_LOSS
    for y in range(1, LIFESPAN + 1):
        npv += (revenue_annual * (decay_rate ** (y - 1)) - total_opex_annual) / ((1 + DISCOUNT_RATE) ** y)
    return [er, sc, npv]

In [10]:
# ==================== 5~8. 场景循环批量优化与结果输出 ====================
from pymoo.core.variable import Real, Integer
from pymoo.core.problem import Problem
from pymoo.optimize import minimize
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

# 优化算法超参数
CROSSOVER_RATE = 0.85
MUTATION_RATE = 0.166
POP_SIZE = 80
N_GENERATIONS = 150
SEED = 41

SCENARIOS = ['Multi', 'HR_II', 'HR_I']

for current_scenario in SCENARIOS:
    print(f"\n{'='*60}")
    print(f"🚀 开始执行情景优化: {current_scenario}")
    print(f"{'='*60}")

    # 1. 设置当前场景约束边界
    if current_scenario == 'Multi': FLOOR_BOUNDS = (4, 9)
    elif current_scenario == 'HR_II': FLOOR_BOUNDS = (10, 18)
    elif current_scenario == 'HR_I': FLOOR_BOUNDS = (19, 33)

    variables = {
        "FloorNum": Integer(bounds=FLOOR_BOUNDS),
        "FAR_pct": Real(bounds=(0.0, 1.0)),
        "depth_pct": Real(bounds=(0.0, 1.0)),
        "front_pct": Real(bounds=(0.0, 1.0)),
        "rotation": Real(bounds=(-5.0, 5.0)),
        "MainTypeIdx": Integer(bounds=(0, 2))
    }
    var_names = ["FloorNum", "FAR_pct", "depth_pct", "front_pct", "rotation", "MainTypeIdx"]

    # 动态定义 Problem 类 (利用闭包传入当前 bounds)
    class DynamicBuildingProblem(Problem):
        def __init__(self, current_bounds, vars_dict, v_names):
            self.current_bounds = current_bounds
            _xl = np.array([vars_dict[name].bounds[0] for name in v_names])
            _xu = np.array([vars_dict[name].bounds[1] for name in v_names])
            super().__init__(n_var=len(v_names), n_obj=3, xl=_xl, xu=_xu, vars=vars_dict)

        def _evaluate(self, X, out, *args, **kwargs):
            designs = [resolve_design(x, self.current_bounds) for x in X]
            svf_batch = predict_svf_batch([svf_geom_row(d) for d in designs])
            res = [objective_func(x, self.current_bounds, svf_val=svf_batch[k]) for k, x in enumerate(X)]
            out["F"] = -np.array(res)

    # 2. 运行 NSGA-II 算法
    algorithm = NSGA2(
        pop_size=POP_SIZE,
        crossover=SBX(prob=CROSSOVER_RATE, eta=20),
        mutation=PM(prob=MUTATION_RATE, eta=20),
        eliminate_duplicates=True
    )

    problem_instance = DynamicBuildingProblem(FLOOR_BOUNDS, variables, var_names)
    res = minimize(problem_instance, algorithm, ('n_gen', N_GENERATIONS), seed=SEED, verbose=True)

    # 3. 结果提取与后处理
    ind_names = ['FloorNum', 'FAR_pct', 'depth_pct', 'front_pct', 'Rotation_raw', 'MainTypeIdx']
    obj_names = ['ER_Tons', 'SC', 'System_NPV']
    faces = ['Roof', 'East', 'South', 'West', 'North']

    X_mat = (np.array([[x[name] for name in ind_names] for x in res.X], dtype=float)
             if isinstance(res.X[0], dict) else np.array(res.X, dtype=float))
    df_res = pd.DataFrame(X_mat, columns=ind_names)

    def get_feature_details_local(x):
        b_feat, c_feat, h_amt = get_full_feature_vector(x, FLOOR_BOUNDS)
        return [b_feat[0], b_feat[1], h_amt, b_feat[3], b_feat[4], b_feat[5], b_feat[6], c_feat[1]]

    feature_array = np.array([get_feature_details_local(x) for x in res.X])
    detail_cols = ['FAR', 'Height', 'Building_Amount', 'Shape_Factor', 'Depth', 'Frontage', 'Real_Rotation', 'Code_Encoded']
    df_res[detail_cols] = feature_array

    for i, name in enumerate(obj_names):
        df_res[name] = -res.F[:, i]

    df_res['Layout_Style'] = df_res['MainTypeIdx'].astype(int).map({0: 'Row', 1: 'Courtyard', 2: 'Cluster'})
    df_res['Height_Class'] = current_scenario # 直接标记当前情景类别

    for i in range(len(df_res)):
        r = df_res.iloc[i]
        p_ind = X_mat[i]
        b_feat, c_feat, h_amt = get_full_feature_vector(p_ind, FLOOR_BOUNDS)
        h_floor_real = int(np.round(p_ind[0]))
        single_area = b_feat[4] * b_feat[5]
        f_far, f_height, f_amount, f_sf, f_depth, f_front, f_rot = b_feat
        h_main_idx_loop, h_code_loop = c_feat

        for f in faces:
            coeff = real_config['PV_COEFF']['roof'] if f == 'Roof' else real_config['PV_COEFF']['wall']
            if f == 'Roof': area = r['Frontage'] * r['Depth'] * r['Building_Amount'] * coeff
            elif f in ['East', 'West']: area = r['Depth'] * r['Height'] * r['Building_Amount'] * coeff
            else: area = r['Frontage'] * r['Height'] * r['Building_Amount'] * coeff
            df_res.loc[i, f'Area_{f}'] = area

        num_hours = bg_weather_matrix.shape[0]
        bd_loop = (f_front * f_depth * f_amount) / real_config['PLOT_AREA']
        svf_loop = predict_svf(f_far, bd_loop, f_height, f_amount, f_sf, f_depth, f_front, h_main_idx_loop, h_code_loop, f_rot)
        fp = build_feature_pool(num_hours, f_far, f_height, f_amount, f_sf, f_depth, f_front, f_rot, h_main_idx_loop, h_code_loop, svf_loop)
        X_pred = to_model_matrix(fp)
        eui_pred = np.maximum(models['hourly_EUI'].predict(X_pred), 0)
        gen = {t.split('_')[-1]: np.maximum(models[t].predict(X_pred), 0) for t in TARGETS[1:]}
        actual_load_h = eui_pred * (single_area * h_floor_real * h_amt)

        for f in faces:
            face_gen = gen[f.lower()]
            df_res.loc[i, f'Gen_kWh_{f}'] = face_gen.sum()
            df_res.loc[i, f'ER_Tons_{f}'] = face_gen.sum() * (GRID_CARBON_FACTOR - PV_CARBON_FACTOR) / 1e6

        def calc_npv_for_faces(active_faces):
            if not active_faces: return 0.0
            total_gen_h = sum([gen[f.lower()] for f in active_faces])
            total_ann_self = np.sum(np.minimum(actual_load_h, total_gen_h))
            total_ann_export = np.maximum(0, total_gen_h.sum() - total_ann_self)
            revenue = total_ann_self * ELECTRICITY_PRICE + total_ann_export * FEED_IN_TARIFF
            capex = sum([df_res.loc[i, f'Area_{f}'] * (PV_COST_PER_M2_ROOF if f == 'Roof' else PV_COST_PER_M2_WALL) for f in active_faces])
            opex = capex * MAINTENANCE_RATE
            npv = -capex
            decay_rate = 1.0 - EFFICIENCY_LOSS
            for y in range(1, LIFESPAN + 1):
                npv += (revenue * (decay_rate ** (y - 1)) - opex) / ((1 + DISCOUNT_RATE) ** y)
            return npv

        npv_total_all = calc_npv_for_faces(faces)
        for f in faces:
            npv_without = calc_npv_for_faces([x for x in faces if x != f])
            marginal_npv = npv_total_all - npv_without
            df_res.loc[i, f'Marginal_NPV_{f}'] = marginal_npv
            df_res.loc[i, f'Marginal_NPV_per_m2_{f}'] = marginal_npv / (df_res.loc[i, f'Area_{f}'] + 1e-9)
        df_res.loc[i, 'True_System_NPV'] = npv_total_all

    # 4. 打印报告与保存独立 CSV
    df_res.to_csv(os.path.join(RESULT_DIR, f"opt_{current_scenario}_full_report.csv"), index=False)



🚀 开始执行情景优化: Multi
n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |       80 |     31 |             - |             -
     2 |      160 |     52 |  0.0377733645 |         ideal
     3 |      240 |     80 |  0.0281692873 |         ideal
     4 |      320 |     80 |  0.0138076816 |             f
     5 |      400 |     80 |  0.0191571512 |         ideal
     6 |      480 |     80 |  0.0173636064 |         ideal
     7 |      560 |     80 |  0.0160787899 |         nadir
     8 |      640 |     80 |  0.0113388670 |         ideal
     9 |      720 |     80 |  0.0391026899 |         ideal
    10 |      800 |     80 |  0.0117395267 |         ideal
    11 |      880 |     80 |  0.0070743482 |         nadir
    12 |      960 |     80 |  0.0079546613 |             f
    13 |     1040 |     80 |  0.0161656047 |         ideal
    14 |     1120 |     80 |  0.0057013163 |         nadir
    15 |     1200 |     80 |  0.0094355704 |             f
    16 |     1280 |     80 |  0.01162

In [14]:
# ==================== 批量读取CSV并重新绘制三联图 ====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import seaborn as sns

RESULT_DIR = r'D:\research_paper\photovoltaic_prediction\optimization_result'
save_path = os.path.join(RESULT_DIR, "optimization_plots")
os.makedirs(save_path, exist_ok=True)

SCENARIOS = ['Multi', 'HR_II', 'HR_I']
faces = ['Roof', 'East', 'South', 'West', 'North']

# 1. 全面调大所有全局字体尺寸
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 24,  # 基础字号
    'axes.titlesize': 30,  # 坐标轴标题字号
    'axes.labelsize': 24,  # 坐标轴标签字号
    'xtick.labelsize': 20,  # X轴刻度字号
    'ytick.labelsize': 20,  # Y轴刻度字号
    'legend.fontsize': 22,  # 图例字号
    'mathtext.fontset': 'stix',
    'font.family': 'STIXGeneral'
})


def map_layout_names(x):
    x_str = str(x)
    if 'Row' in x_str: return 'Row-style'
    if 'Court' in x_str: return 'Courtyard'
    if 'Cluster' in x_str: return 'Cluster-style'
    return x_str


# 2. 循环读取三个CSV，分别出图
for current_scenario in SCENARIOS:
    csv_file = os.path.join(RESULT_DIR, f"opt_{current_scenario}_full_report.csv")
    if not os.path.exists(csv_file):
        print(f"⚠️ 找不到 {csv_file}，跳过此情景。")
        continue

    print(f"正在绘制 {current_scenario} 的图表...")
    df_res = pd.read_csv(csv_file)

    fig = plt.figure(figsize=(24, 40))
    gs = gridspec.GridSpec(5, 1, height_ratios=[3.5, 3.5, 1, 1, 1], hspace=0.35)

    df_norm = df_res.copy()
    df_res['Layout_Style'] = df_res['Layout_Style'].apply(map_layout_names)
    df_norm['Layout_Style'] = df_res['Layout_Style']

    # ================= (A) 平行坐标 =================
    ax1 = fig.add_subplot(gs[0])
    plot_cols = ['ER_Tons', 'SC', 'True_System_NPV'] + [f'Marginal_NPV_per_m2_{f}' for f in faces]
    df_norm_plot = (df_norm[plot_cols] - df_norm[plot_cols].min()) / (
                df_norm[plot_cols].max() - df_norm[plot_cols].min() + 1e-9)
    df_norm_plot['Layout_Style'] = df_norm['Layout_Style']
    cmaps = {'Courtyard': plt.cm.Reds, 'Row-style': plt.cm.Blues, 'Cluster-style': plt.cm.Greens}
    norm_color = mcolors.Normalize(vmin=df_res['SC'].min(), vmax=df_res['SC'].max())
    x_ticks = range(len(plot_cols))

    for idx, row in df_norm_plot.iterrows():
        layout = row['Layout_Style']
        sc_real_value = df_res.loc[idx, 'SC']
        color_val = cmaps.get(layout, plt.cm.gray)(norm_color(sc_real_value) * 0.6 + 0.4)
        ax1.plot(x_ticks, row[plot_cols], color=color_val, alpha=0.75, lw=2.5)

    custom_lines = [Line2D([0], [0], color=cmaps['Courtyard'](0.8), lw=4),
                    Line2D([0], [0], color=cmaps['Row-style'](0.8), lw=4),
                    Line2D([0], [0], color=cmaps['Cluster-style'](0.8), lw=4)]
    ax1.set_title(f'(A) Parallel coordinates plot of pareto front ({current_scenario})', fontweight='bold', pad=15,
                  loc='left')
    ax1.set_ylabel('Normalized Score (0-1)', labelpad=15)
    ax1.set_xticks(x_ticks)
    ax1.set_xticklabels(
        ['Total ER', 'Total SC', 'Total NPV', 'NPV Roof', 'NPV East', 'NPV South', 'NPV West', 'NPV North'],
        rotation=15)
    ax1.set_xlim(0, len(plot_cols) - 1)
    ax1.legend(custom_lines, ['Courtyard', 'Row-style', 'Cluster-style'], title='Block Layout',
               bbox_to_anchor=(1.01, 1), loc='upper left')

    # ================= (B) 热力图 =================
    ax2 = fig.add_subplot(gs[1])
    df_res_sorted = df_res.sort_values(by='SC', ascending=False).reset_index(drop=True)
    selected_indices = np.linspace(0, len(df_res_sorted) - 1, 20).astype(int)
    df_heatmap_data = df_res_sorted.iloc[selected_indices].copy()

    heat_cols = ['SC'] + [f'ER_Tons_{f}' for f in faces] + [f'Marginal_NPV_per_m2_{f}' for f in faces]
    sns.heatmap(df_heatmap_data[heat_cols], annot=True, fmt=".1f", cmap="YlGnBu",
                xticklabels=['SC'] + [f'ER ({f})' for f in faces] + [f'NPV/m² ({f})' for f in faces],
                yticklabels=[f"ID {int(idx)}" for idx in df_heatmap_data.index],
                annot_kws={"size": 18}, linewidths=1, linecolor='white', ax=ax2)
    ax2.set_title(f'(B) Heatmap of pareto solutions ({current_scenario})', fontweight='bold', pad=15, loc='left')
    ax2.set_ylabel('Selected Solutions (Sorted by SC)', labelpad=15)
    ax2.tick_params(axis='x', rotation=30)

    # ================= (C) 趋势图 =================
    x_axis = np.arange(len(df_res_sorted))
    trend_sc = df_res_sorted['SC']
    trend_npv = df_res_sorted['True_System_NPV'] / 10000
    trend_er = df_res_sorted['ER_Tons']


    def format_trend_ax(ax, title):
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_title(title, loc='center', y=-0.35, fontsize=24)
        ax.set_xlim(0, len(x_axis) - 1);
        ax.margins(x=0)
        ax.spines['top'].set_visible(False);
        ax.spines['right'].set_visible(False)


    # 截断 Y 轴，放大 SC 的波动趋势
    ax3 = fig.add_subplot(gs[2])
    sc_min, sc_max = trend_sc.min(), trend_sc.max()
    sc_pad = (sc_max - sc_min) * 0.1 if sc_max > sc_min else 0.05
    y_bottom_sc = sc_min - sc_pad

    ax3.fill_between(x_axis, trend_sc, y2=y_bottom_sc, color='#E1F5FE', alpha=0.8)
    ax3.plot(x_axis, trend_sc, color='#0288D1', lw=2)
    ax3.set_ylim(y_bottom_sc, sc_max + sc_pad)
    ax3.set_ylabel('Ratio')
    format_trend_ax(ax3, '(C) FV 1: SC (Self-Consumption)')

    ax4 = fig.add_subplot(gs[3])
    ax4.fill_between(x_axis, trend_npv, color='#E1F5FE', alpha=0.8);
    ax4.plot(x_axis, trend_npv, color='#0288D1', lw=2)
    ax4.set_ylabel('10k CNY');
    format_trend_ax(ax4, '(D) FV 2: True System NPV')

    ax5 = fig.add_subplot(gs[4])
    ax5.fill_between(x_axis, trend_er, color='#E1F5FE', alpha=0.8);
    ax5.plot(x_axis, trend_er, color='#0288D1', lw=2)
    ax5.set_ylabel('Tons');
    format_trend_ax(ax5, '(E) FV 3: Emission Reduction (ER)')

    # 动态保存文件命名，防止覆盖
    fig_name = f"comprehensive_analysis_{current_scenario}.png"
    plt.savefig(os.path.join(save_path, fig_name), dpi=300, bbox_inches='tight')
    plt.close(fig)

print("✅ 三种情景的最新版图表全部更新完毕！")

正在绘制 Multi 的图表...
正在绘制 HR_II 的图表...
正在绘制 HR_I 的图表...
✅ 三种情景的最新版图表全部更新完毕！
